# GCI study: model-swap regeneration test on agent-authored code

This notebook measures the **Generative Coupling Index (GCI)** for the field study in Section IX of the paper.
Run the cells **in order, top to bottom**, with **Shift+Enter**. Wait for each cell to finish (the spinner stops)
before running the next one.

| Step | What it does | Time | Cost |
|---|---|---|---|
| 1 | Upload the replication package and install libraries | 2 min | free |
| 2 | Enter your Anthropic and Ollama keys (hidden input) | 1 min | free |
| 3 | Choose the model IDs (Claude + one Ollama Cloud model) | 1 min | free |
| 4 | Check every key and model with one call each | 1 min | < $0.01 |
| 5 | Download the AIDev dataset | 10–20 min | free |
| 6 | Extract the functions (blocks) to test (pilot: Claude Code only) | 5–10 min | free |
| 7 | **Pilot**: 30 Claude Code blocks, two prompt versions | 10–20 min | about $2 |
| 8 | Look at the pilot result and decide | — | — |
| 9 | **Full run**: 250 blocks | 1–2 h | about $6–10 |
| 9b | Results: tables, figures, draft text | 1 min | free |
| 10 | Download the results and send them back | 1 min | free |

**If the session disconnects:** reconnect, then re-run steps 1–3 and continue where you stopped.
API answers are cached in the results folder, so nothing you already paid for is paid again,
as long as you have not deleted the runtime. To keep results safe across disconnects, use the optional
Google Drive cell in step 1.

## Step 1: Upload the replication package and install libraries

Either upload `reuse-metrics-replication.zip` through the Files panel on the left first, or run the cell and use
the **Choose Files** button that appears. Use the latest zip (7 November, with the Ollama Cloud provider).

In [ ]:
import os, glob, zipfile
from google.colab import files

zips = sorted(glob.glob("/content/**/reuse-metrics*.zip", recursive=True), key=os.path.getmtime)
if not zips:
    print("Choose reuse-metrics-replication.zip:")
    uploaded = files.upload()
    zips = [os.path.abspath(next(iter(uploaded)))]
zip_path = zips[-1]
print("Using:", zip_path)
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content")
os.chdir("/content")

!pip -q install -e /content/reuse-metrics pandas pyarrow huggingface_hub "anthropic>=2" google-genai openai requests
src = open("/content/reuse-metrics/studies/swap_test_pilot/llm_oracle.py").read()
assert "OllamaProvider" in src and "api/chat" in src, "Old scripts: upload the latest gci_scripts.zip (or reuse-metrics-replication.zip) and re-run this cell"
print("Ready.")

**Optional (recommended for the full run):** save results to Google Drive so they survive a disconnect.
Run this cell only if you want that; it asks you to allow access to your Drive.

In [ ]:
USE_DRIVE = False   # change to True, then run this cell

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/gci_study"
else:
    OUT_ROOT = "/content"
os.makedirs(OUT_ROOT, exist_ok=True)
print("Results will be written under:", OUT_ROOT)

## Step 2: Enter your API keys (Anthropic and Ollama)

Each prompt hides what you type. Paste the key and press Enter. The keys stay in this session only and are not saved
in the notebook. **Do not share this notebook with your keys typed into a cell.**

In [ ]:
import re
from getpass import getpass
KEYS = [("ANTHROPIC_API_KEY", "Anthropic (Claude)", "sk-ant-"),
        ("OLLAMA_API_KEY",    "Ollama (cloud)",     "")]
for var, label, prefix in KEYS:
    while not os.environ.get(var, "").strip():
        hint = f" (starts with {prefix})" if prefix else ""
        val = getpass(f"Paste your {label} key{hint} and press Enter: ").strip()
        if not val:
            print("  Nothing was pasted, try again.")
            continue
        if prefix and not val.startswith(prefix):
            print(f"  Warning: this does not start with {prefix}. Check you pasted the {label} key.")
        if var == "OLLAMA_API_KEY":
            val = re.sub(r"^\s*(Bearer|OLLAMA_API_KEY)[\s:=_-]*", "", val, flags=re.I).strip()   # drop a pasted label
        os.environ[var] = val
for var, label, _ in KEYS:
    v = os.environ[var]
    print(f"{label:20s} set: {v[:6]}...{v[-4:]}  ({len(v)} characters)")

To **re-enter** a key (for example after pasting the wrong one), run this cell, then run the cell above again.

In [ ]:
for var in ["ANTHROPIC_API_KEY", "OLLAMA_API_KEY"]:
    os.environ.pop(var, None)
print("Keys cleared. Run the key cell above again.")

## Step 3: Choose the models (Claude + one Ollama Cloud model)

Two Claude models and one open-weights model served by Ollama Cloud, so the swap test crosses two vendors.
The first cell asks Ollama which models your key can use and picks a coding model from a different maker than Anthropic.
Look at the list it prints; if you prefer another model, set `OLLAMA_MAIN` yourself in the second cell.
The *strong* Claude model is also used for the self-regeneration control of the Claude Code agent.

In [ ]:
import requests
r = requests.get("https://ollama.com/api/tags", headers={"Authorization": "Bearer " + os.environ["OLLAMA_API_KEY"]}, timeout=60)
print("HTTP", r.status_code)
if r.status_code == 401:
    raise SystemExit("Ollama rejected the key: clear it (step 2, second cell) and enter it again.")
AVAILABLE = sorted(m["name"] for m in r.json().get("models", []))
print(len(AVAILABLE), "models available to your key:")
for n in AVAILABLE: print("  ", n)

In [ ]:
# --- Models (order matters: GCI is reported for the first 1, 2, 3 ... models) ---
CLAUDE_SMALL = "claude-haiku-5-5"
CLAUDE_LARGE = "claude-sonnet-5-5"

OLLAMA_MAIN   = ""   # type a name from the list above to override the automatic choice
OLLAMA_SECOND = ""   # optional 4th model from ANOTHER maker (adds about 750 calls); leave "" to skip
VENDOR_OF = [("qwen", "Alibaba"), ("gpt-oss", "OpenAI"), ("deepseek", "DeepSeek"), ("glm", "Z.ai"),
             ("nemotron", "NVIDIA"), ("kimi", "Moonshot"), ("llama", "Meta"), ("mistral", "Mistral"), ("devstral", "Mistral")]
if not OLLAMA_MAIN:
    for pref in ["qwen3.8", "qwen3-coder", "gpt-oss:120b", "deepseek", "qwen3", "gpt-oss"]:
        hit = [n for n in AVAILABLE if pref in n.lower()]
        if hit:
            OLLAMA_MAIN = hit[0]; break
assert OLLAMA_MAIN in AVAILABLE, "Set OLLAMA_MAIN to a name printed in the list above"
maker = lambda n: next((v for k, v in VENDOR_OF if k in n.lower()), "Ollama")
OLLAMA_VENDOR = maker(OLLAMA_MAIN)

MODELS = [f"anthropic:{CLAUDE_SMALL}", f"anthropic:{CLAUDE_LARGE}", f"ollama:{OLLAMA_MAIN}"]
VENDORS = f"anthropic=Anthropic ollama:{OLLAMA_MAIN}={OLLAMA_VENDOR}"
if OLLAMA_SECOND:
    assert OLLAMA_SECOND in AVAILABLE, "OLLAMA_SECOND must be a name printed in the list above"
    MODELS.append(f"ollama:{OLLAMA_SECOND}"); VENDORS += f" ollama:{OLLAMA_SECOND}={maker(OLLAMA_SECOND)}"

# Control-step proxy: only Claude Code blocks have their vendor in the model set.
# Copilot, Codex, Cursor, Devin and Jules blocks get "no_control" verdicts and are reported separately.
ORIGIN = [f"Claude_Code=anthropic:{CLAUDE_LARGE}"]

MODELS_ARG, ORIGIN_ARG = " ".join(MODELS), " ".join(ORIGIN)
print("Vendors:", VENDORS); print(MODELS_ARG); print(ORIGIN_ARG)

## Step 4: Check every key and model (one short call each)

Every line must start with **OK**. A **FAIL** line means that key or model ID is wrong:
fix it in step 2 or 3, re-run that cell, then re-run this one.

In [ ]:
!cd /content/reuse-metrics/studies/swap_test_pilot && python smoke_test_keys.py {MODELS_ARG}

## Step 5: Download the AIDev dataset

Two files from Hugging Face: `pull_request.parquet` (small) and `pr_commit_details.parquet` (about 12 GB).
**There is no progress bar**: the cell is working while the spinner turns. A warning about
"unauthenticated requests to the HF Hub" is normal and can be ignored.

In [ ]:
from huggingface_hub import hf_hub_download
for t in ["pull_request", "pr_commit_details"]:
    print(t, "->", hf_hub_download("hao-li/AIDev", f"{t}.parquet", repo_type="dataset", revision="v5"))

To check the download is moving, run this in a **new** cell while step 5 runs (the size should grow):

In [ ]:
!du -sh /root/.cache/huggingface/hub/datasets--hao-li--AIDev 2>/dev/null || echo "not started yet"

## Step 6: Extract the functions to test

Finds whole functions that the agents added in the 105 study repositories.
- **Pilot:** 30 functions written by Claude Code that have a docstring. Only Claude Code blocks can be *controlled*
  (checked for whether the PR text is enough to regenerate them), so the pilot looks at exactly those.
- **Full run:** 250 functions, with up to 150 from Claude Code and the rest spread over the other agents.
  Only documented functions are used, because the docstring is part of the specification given to the models.

In [ ]:
PILOT, FULL = f"{OUT_ROOT}/gci_pilot", f"{OUT_ROOT}/gci"
REPOS = "reuse-metrics/results/field_study/pooled_per_repo.csv"
!python -W ignore reuse-metrics/studies/swap_test_pilot/extract_blocks.py --repos {REPOS} --n-blocks 30 --seed 2 --only-agents Claude_Code --require-docstring --out {PILOT}/blocks.jsonl
!python -W ignore reuse-metrics/studies/swap_test_pilot/extract_blocks.py --repos {REPOS} --n-blocks 250 --seed 1 --agent-quota Claude_Code=150 --require-docstring --out {FULL}/blocks.jsonl

## Step 7: Pilot run (30 Claude Code blocks, two prompt versions, about $2)

The same 30 blocks are run twice: once with the **PR text only** and once with the PR text **plus the function's own
docstring** as a specification. Progress lines appear every 10 blocks.
Messages like `retry in 4s` are normal: a vendor asked the script to slow down and it waits automatically.

In [ ]:
# --sleep 1.1 keeps the request rate low enough that Ollama Cloud and Anthropic rarely refuse calls
COMMON = f"--models {MODELS_ARG} --origin {ORIGIN_ARG} --theta 0.3 0.4 0.5 --R 3 --q 0.34 --sleep 1.1 --per-repo {REPOS}"
!python -W ignore reuse-metrics/studies/swap_test_pilot/gci_run.py --blocks {PILOT}/blocks.jsonl {COMMON} --prompt-level pr  --distance token --out {PILOT}_pr
!python -W ignore reuse-metrics/studies/swap_test_pilot/gci_run.py --blocks {PILOT}/blocks.jsonl {COMMON} --prompt-level doc --distance token --out {PILOT}_doc

## Step 7b: Re-score the pilot with a relaxed distance (free)

The strict token distance counts a renamed variable as a difference, so two functions that do the same thing under
different local names look far apart. The **alpha** distance renames the function and every local name to positional ids
before comparing. This cell re-scores the same model answers (read from the cache, no API calls, no cost).

In [ ]:
for level in ["pr", "doc"]:
    !python -W ignore reuse-metrics/studies/swap_test_pilot/gci_run.py --blocks {PILOT}/blocks.jsonl {COMMON} --prompt-level {level} --distance alpha --cache {PILOT}_{level}/cache.jsonl --out {PILOT}_{level}_alpha | tail -n 3

## Step 7c: More samples per model (about $1-2)

At their default temperature the Claude models give a different function on each try, so three samples may all miss a
function that a fourth would have reproduced. This cell draws **five** samples per model and takes the best one
(`--R 5 --q 0.2`: a function counts as regenerable if at least one of five samples is within θ).
The first three samples per model are already cached, so only two more per model and block are paid for.

In [ ]:
R5 = "--R 5 --q 0.2"
!python -W ignore reuse-metrics/studies/swap_test_pilot/gci_run.py --blocks {PILOT}/blocks.jsonl {COMMON} {R5} --prompt-level doc --distance token --cache {PILOT}_doc/cache.jsonl --out {PILOT}_doc_r5 | tail -n 3
!python -W ignore reuse-metrics/studies/swap_test_pilot/gci_run.py --blocks {PILOT}/blocks.jsonl {COMMON} {R5} --prompt-level doc --distance alpha --cache {PILOT}_doc/cache.jsonl --out {PILOT}_doc_r5_alpha | tail -n 3

## Step 8: Look at the pilot before spending more

The question is whether Claude can regenerate **its own** function from the prompt at all (the control).
If it cannot, the prompt is too weak and a "coupled" verdict would only mean "under-specified".
Read the table:
1. **under_specified_share**: below about 0.5 is workable. Near 1.0 means the prompt does not determine the function.
2. **d_self median**: the typical distance between Claude's own regeneration and the original (0 = identical, 1 = unrelated).
3. **theorem3_monotone** is `true`.

Choose the setting for the full run: prompt (`pr`/`doc`), distance (strict, or `_alpha`) and samples (`doc_r5` = five samples).
Report the others as robustness checks. Do not keep adjusting the metric after this step: a low under-specified share is not the goal.
If a large share of functions cannot be regenerated even by their own model, that is itself a finding (the PR text does not
determine the code), and GCI is then reported for the regenerable part, with the share stated. If both are still above 0.7,
stop and send me this table: the threshold θ or the prompt needs another look before spending more.

In [ ]:
import json, pandas as pd
rows = []
for level in ["pr", "doc", "pr_alpha", "doc_alpha", "doc_r5", "doc_r5_alpha"]:
    s = json.load(open(f"{PILOT}_{level}/summary.json"))
    d = s.get("d_self_controlled", {})
    for th, r in s["by_theta"].items():
        rows.append({"version": level, "theta": th, "controlled_blocks": r["n_with_control"],
                     "under_specified_share": r.get("under_specified_share"),
                     "coupled_share_given_control": r.get("coupled_block_share_given_control"),
                     "theorem3_monotone": r["theorem3_monotone"],
                     "d_self quantiles 10/25/50/75/90": d.get("quantiles_10_25_50_75_90")})
display(pd.DataFrame(rows))
print("API calls (only doc_r5 should be non-zero beyond the first two runs):", {l: json.load(open(f'{PILOT}_{l}/summary.json'))["api_calls_this_run"] for l in ["pr", "doc", "pr_alpha", "doc_alpha", "doc_r5", "doc_r5_alpha"]})

## Step 9: Full run (250 blocks, prompt version chosen in step 8)

Only after the pilot looks right. Set `LEVEL` to the version with the lower under-specified share. This takes one to two hours: **keep the browser tab open**.
If it disconnects, reconnect, re-run steps 1–3 (and the Drive cell if used), then re-run this cell:
it resumes from the cache.

In [ ]:
LEVEL = "doc"      # "pr" or "doc", from step 8
RFULL = "--R 3 --q 0.34"   # or R5 = "--R 5 --q 0.2" if step 7c helped
DIST  = "alpha"    # "token" (strict) or "alpha" (local names renamed), from step 8
!python -W ignore reuse-metrics/studies/swap_test_pilot/gci_run.py --blocks {FULL}/blocks.jsonl {COMMON} {RFULL} --prompt-level {LEVEL} --distance {DIST} --out {FULL}

### Step 9 robustness: the same run with the other distance (free)

Re-scores the full run's cached answers with the distance you did **not** choose, so the paper can report both.

In [ ]:
OTHER = "token" if DIST == "alpha" else "alpha"
!python -W ignore reuse-metrics/studies/swap_test_pilot/gci_run.py --blocks {FULL}/blocks.jsonl {COMMON} {RFULL} --prompt-level {LEVEL} --distance {OTHER} --cache {FULL}/cache.jsonl --out {FULL}_{OTHER} | tail -n 3
!python -W ignore reuse-metrics/studies/swap_test_pilot/gci_analyse.py --run {FULL}_{OTHER} --models {MODELS_ARG} --vendors {VENDORS} --theta-main 0.4 --out {FULL}_{OTHER}/analysis | tail -n 4

## Step 9b: Results, tables and figures

Turns the full run into everything the paper needs, in `gci/analysis/`:
- `results.json`: every number, with 95% intervals from a bootstrap over repositories
- `tab_gci_overview.tex`, `tab_gci_agents.tex`, `tab_gci_models.tex`: LaTeX tables
- `fig_gci_distances.pdf`, `fig_gci_nested.pdf`, `fig_gci_agents.pdf`, `fig_gci_robustness.pdf`: figures (PNG copies too)
- `section_gci_draft.tex`: a draft Section IX subsection with the numbers filled in

The figures are shown below the cell. Takes about a minute (the bootstrap).

In [ ]:
!python -W ignore reuse-metrics/studies/swap_test_pilot/gci_analyse.py --run {FULL} --models {MODELS_ARG} --vendors {VENDORS} --theta-main 0.4 --out {FULL}/analysis

from IPython.display import Image, display
for f in ["fig_gci_distances", "fig_gci_nested", "fig_gci_agents", "fig_gci_robustness"]:
    print(f); display(Image(f"{FULL}/analysis/{f}.png", width=650))
print(open(f"{FULL}/analysis/tab_gci_overview.tex").read())

The same analysis can be run on the pilot alone (useful for a first look; intervals will be wide):

In [ ]:
!python -W ignore reuse-metrics/studies/swap_test_pilot/gci_analyse.py --run {PILOT}_doc --models {MODELS_ARG} --vendors {VENDORS} --theta-main 0.4 --boot 500 --out {PILOT}_doc/analysis

## Step 10: Download the results

Creates `gci_results.zip` with the pilot, the full run and all tables and figures, and downloads it to your
computer. Send that zip back for Section IX. It contains no API keys.

In [ ]:
!cd {OUT_ROOT} && zip -q -r /content/gci_results.zip gci_pilot_pr gci_pilot_doc gci_pilot_pr_alpha gci_pilot_doc_alpha gci_pilot_doc_r5 gci_pilot_doc_r5_alpha gci gci_token gci_alpha -x "*/cache.jsonl"
!cd {OUT_ROOT} && zip -q -r /content/gci_cache.zip gci_pilot_pr/cache.jsonl gci_pilot_doc/cache.jsonl gci/cache.jsonl 2>/dev/null; true
files.download("/content/gci_results.zip")
print("Also keep gci_cache.zip (the raw model answers) for the replication package:")
files.download("/content/gci_cache.zip")

## If something goes wrong

| What you see | What to do |
|---|---|
| `No module named ...` | Re-run step 1, then **Runtime → Restart session**, then steps 1–3 again |
| `FAIL` in step 4 | Wrong key or model ID. Fix step 2 or 3 and re-run |
| `... API_KEY is empty: run the key cell again` | Step 2 was skipped after a restart, or nothing was pasted. Run step 2 again |
| Many `retry in ...` lines | Normal under rate limits. If it never moves on, change `--sleep 1.1` to `--sleep 2` in step 7 |
| Ollama `HTTP 429` or a usage-limit message | You reached your Ollama plan's limit. Wait for it to reset (check your Ollama usage page) and re-run the same cell: it resumes from the cache |
| `no credit left` (Anthropic) | Add credit on that vendor's billing page, then re-run the same cell |
| Ollama `404 model_not_found` | The model name is not in the step 3 list. Set `OLLAMA_MAIN` to a printed name and re-run step 3 |
| Any other red error | Copy the last 15 lines and send them back |